## Rates table

In [51]:
# Step 1: Filter and Clean Invoice Data
import pandas as pd


# 🔧 Configure which sites to process
#selected_sites = ["DIT", "SPN", "SPCP","SPW","SPT","SPHU","SPTM","PVF","SPJ","CCS","SPB","SPL","SPLV","CCSG","SPCB","SPWV","FSU","SPK","SPLA","SPD","SPTG","KFC"]  # Example: update these as needed
selected_sites =['SPW','SPT','SPJ','SPN']

#selected_sites =['SPJ']

focus_columns = [
    "invoice_id", "site",'invoice_commodity_quantity', "invoice_commodity_group", "invoice_commodity_description",
    "location", "model", "unit", "rate_unit", "freight_class", "applied_rate",
    "shipment_type", "realistic_optimal_method", "xgs_rate", "historical_rate"
]

# Load the invoice input data
invoice_path = "invoice_input_data_all.xlsx"  # Update path if needed
invoice_df = pd.read_excel(invoice_path)
print("Input file from model has",invoice_df['invoice_id'].nunique())


# get invoces that have freight greater than zero
invoice_df = invoice_df[invoice_df["rate_ratio_normal_outlier"]!= 'MISSING'] # Key update here to remove rate_ratio_normal_outlier

print("Output analysis files has",invoice_df['invoice_id'].nunique())

# Only modelled for Georgia source of Georgia mill rates
invoice_georgia_df = invoice_df[invoice_df['model'] == True]
print("Output analysis file for Georga Mills has",invoice_georgia_df['invoice_id'].nunique())


# Get all modelled invoices for histotrical estimate
invoice_all_df = invoice_df


invoice_georgia_df = invoice_georgia_df[focus_columns]
invoice_all_df = invoice_all_df[focus_columns]


invoice_georgia_df["invoice_commodity_description"] = invoice_georgia_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

invoice_all_df["invoice_commodity_description"] = invoice_all_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

# Filter input invoices to selected sites
invoice_georgia_df = invoice_georgia_df[invoice_georgia_df["site"].isin(selected_sites)]
invoice_all_df = invoice_all_df[invoice_all_df["site"].isin(selected_sites)]



Input file from model has 17846
Output analysis files has 13813
Output analysis file for Georga Mills has 10300


C:\Users\nzhuw\AppData\Local\Temp/ipykernel_25336/1135092565.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invoice_all_df["invoice_commodity_description"] = invoice_all_df["invoice_commodity_description"].apply(


In [52]:
# We are calculating 3 core variables here:
# invoice_df_mills # this is for georgia_mill_rates and georgia_xgs_rates
# invoice_df_hist #this is for historical_market_rates mills and distributors

Historical market rates have to two variables historical_all_rates and historical_georgia_rates

XGS rates will have the historical_xgs_rates and 2024_xgs_rates

We will recommended a market rate recommended_market_rate


## Functions

In [53]:
# 📌 Import necessary libraries for data processing and file handling
from numpy import average

def summarize_invoice_data(df: pd.DataFrame, group_cols: list[str], column_prefix: str = "") -> pd.DataFrame:
    # Filter required fields
    filtered = df[
        df["freight_class"].notna() &
        df["historical_rate"].notna() &
        df["xgs_rate"].notna() &
        df["invoice_commodity_quantity"].notna()
    ][[
        "site", 
        "rate_unit", 
        "invoice_commodity_group", 
        # "invoice_commodity_description",
        "freight_class", 
        "historical_rate",
        "xgs_rate",
        "invoice_commodity_quantity"
    ]].copy()

    grouped = filtered.groupby(group_cols)

    # Quantiles
    summary_avg = grouped.agg(
        historical_xgs_median = ("xgs_rate", "median"),
        historical_xgs_q3 = ("xgs_rate", lambda x: x.quantile(0.75)),
        
        historical_market_median = ("historical_rate", "median"),
        historical_market_q3 = ("historical_rate", lambda x: x.quantile(0.75)),
    )

    # Rename with prefix
    summary_avg = summary_avg.rename(columns={
        "historical_xgs_q3": f"{column_prefix}historical_xgs_q3",
        "historical_xgs_median": f"{column_prefix}historical_xgs_median",
        "historical_market_q3": f"{column_prefix}historical_market_q3",
        "historical_market_median": f"{column_prefix}historical_market_median"
    })


    # Weighted averages
    def compute_wavg(grp):
        return pd.Series({
            f"{column_prefix}historical_market_wavg": average(grp["historical_rate"], weights=grp["invoice_commodity_quantity"]),
            f"{column_prefix}historical_xgs_wavg": average(grp["xgs_rate"], weights=grp["invoice_commodity_quantity"])
        })

    summary_wavg = grouped.apply(compute_wavg)

    # Combine
    summary = pd.concat([summary_avg, summary_wavg], axis=1)
    return summary


In [54]:
# 📌 Create pivot table to summarize data by specified index and columns

index_cols = ["site", "rate_unit", "invoice_commodity_group",]
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
def safe_pivot(metric_col, source_name,data):
    pivoted = data.pivot_table(
        index=index_cols,
        columns="freight_class",
        values=metric_col,
        aggfunc="first"  # or "mean" if multiple values should be averaged
    ).reset_index()

    # Ensure all freight class columns are present
    for fc in freight_classes:
        if fc not in pivoted.columns:
            pivoted[fc] = None

    # Add required template columns
    pivoted.rename(columns={
        "rate_unit": "unit",
        "invoice_commodity_group": "commodity_group",
    }, inplace=True)
    pivoted["site_description"] = pivoted["site"]
    pivoted["unitclass"] = pivoted["unit"].apply(lambda x: "Weight" if x == "CWT" else "Area")
    pivoted["source"] = source_name

    # Reorder
    ordered_cols = ["site_description", "site", "unit", "unitclass", "commodity_group", ] + freight_classes + ["source"]
    return pivoted[ordered_cols]

In [55]:
def apply_multiplier_to_group(df, group_col, group_val, multiplier):
    """
    Multiplies all numeric columns in the DataFrame for rows matching a specific group value.

    Parameters:
    - df (pd.DataFrame): The input DataFrame
    - group_col (str): Column to filter on (e.g., "invoice_commodity_group")
    - group_val (str): Value to match in the group column (e.g., "1VNL")
    - multiplier (float): Value to multiply matching rows by (e.g., 1.2)

    Returns:
    - pd.DataFrame: Updated DataFrame with scaled numeric values
    """
    df = df.copy()  # avoid modifying original
    numerical_cols = df.select_dtypes(include='number').columns
    mask = df[group_col] == group_val
    df.loc[mask, numerical_cols] = df.loc[mask, numerical_cols] * multiplier
    print(f"✅ Multiplied numeric columns for {group_col} == '{group_val}' by {multiplier}.")
    return df


In [56]:
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
ordered_cols = ["site_description", "site", "unit", "unitclass", "commodity_group"] + freight_classes + ["source"]


def prepare_freight_dataframe(df, freight_classes, ordered_cols, source_label=None):
    """
    Ensures a DataFrame has all required freight class columns and matches a specific column order.

    Parameters:
    - df (pd.DataFrame): Input DataFrame to prepare.
    - freight_classes (list): List of freight class column names to enforce.
    - ordered_cols (list): Final column order (must include freight_classes inside).
    - source_label (str, optional): Value to set in the 'source' column, if provided.

    Returns:
    - pd.DataFrame: Cleaned and reordered DataFrame.
    """
    df = df.copy()

    # Add missing freight class columns
    for col in freight_classes:
        if col not in df.columns:
            df[col] = None

    # Add or update source column
    if source_label is not None:
        df["source"] = source_label

    # Ensure all ordered_cols exist (fill missing with None)
    for col in ordered_cols:
        if col not in df.columns:
            df[col] = None

    # Reorder columns
    df = df[ordered_cols]

    return df


In [57]:
# Constants for rate adjustment
XGS_RATE_DISCOUNT = 0.06
XGS_FUEL_SURCHARGE = 0.3
XGS_LTL_REBATE = 0.1
STARNET_REBATE = 0.025


# Adjustment function
def adjust_rate(rate):
    inflation_rate = rate / (1 + XGS_RATE_DISCOUNT)
    fsc_rate = inflation_rate * (1 + XGS_FUEL_SURCHARGE)
    xgs_rebate = inflation_rate * XGS_LTL_REBATE
    star_net_rebate = (inflation_rate - xgs_rebate) * STARNET_REBATE
    final_rate = fsc_rate - xgs_rebate - star_net_rebate
    return final_rate






In [58]:
def scale_freight_rates_by_condition(df, commodity, unit, freight_class_cols, multiplier):
    """
    Scales freight rate columns by a given multiplier for rows matching a specific commodity and unit.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing freight rate columns.
    - commodity (str): The value in 'commodity_group' to match (e.g. "1VNL").
    - unit (str): The value in 'unit' column to match (e.g. "CWT").
    - freight_class_cols (list): List of freight class columns to scale.
    - multiplier (float): Value to multiply the freight class columns by.

    Returns:
    - pd.DataFrame: The modified DataFrame with rates scaled accordingly.
    """
    df = df.copy()
    mask = (df["commodity_group"] == commodity) & (df["unit"] == unit)
    df.loc[mask, freight_class_cols] = df.loc[mask, freight_class_cols] * multiplier
    print(f"✅ Scaled {commodity} with unit {unit} by {multiplier} for freight classes.")
    return df


In [59]:
def append_invoice_freight_matrix(invoice_df, combined_df, freight_classes, site_description="Itasca"):
    """
    Appends a matrix of invoice counts by freight class and commodity group per site to the combined DataFrame.

    Parameters:
    - invoice_df (pd.DataFrame): DataFrame with at least ['site', 'invoice_commodity_group', 'freight_class', 'invoice_id']
    - combined_df (pd.DataFrame): The master DataFrame to which the result will be appended.
    - freight_classes (list): List of freight class columns to standardize across all sites.
    - site_description (str): Default description to assign to 'site_description' column.

    Returns:
    - pd.DataFrame: Updated combined DataFrame with appended freight class matrix rows.
    """
    # Count unique invoice_ids per site, commodity, and freight class
    invoice_counts = invoice_df.groupby(
        ["site", "invoice_commodity_group", "freight_class"]
    )["invoice_id"].nunique().reset_index(name="invoice_count")

    invoice_matrix_list = []

    for site in invoice_counts["site"].unique():
        site_df = invoice_counts[invoice_counts["site"] == site]

        # Pivot to freight class matrix
        matrix = site_df.pivot_table(
            index="invoice_commodity_group",
            columns="freight_class",
            values="invoice_count",
            fill_value=0
        )

        # Ensure all freight classes are represented
        matrix = matrix.reindex(columns=freight_classes, fill_value=0).reset_index()

        # Add required metadata
        matrix["site_description"] = site_description
        matrix["site"] = site
        matrix["unit"] = None
        matrix["unitclass"] = None
        matrix["commodity_group"] = None
        matrix["source"] = "invoice_counts"

        # Arrange columns in desired order
        ordered_cols = [
            "site_description", "site", "unit", "unitclass", "commodity_group"
        ] + freight_classes + ["source"]
        final_matrix = matrix[ordered_cols]

        invoice_matrix_list.append(final_matrix)

    # Combine and append
    invoice_matrix_final = pd.concat(invoice_matrix_list, ignore_index=True)
    combined_df = pd.concat([combined_df, invoice_matrix_final], ignore_index=True)

    print("✅ Appended invoice freight class matrices per site to combined_df.")
    return combined_df


In [60]:
import pandas as pd

def generate_recommendation_rows(
    merged_df,
    freight_class_cols,
    median_sources,
    wavg_sources,
    q3_sources,
    multiplier=1.06
):
    """
    Generates recommended rows (median, wavg, q3) and ranked rows (min, middle, max) for each group in merged_df.

    Parameters:
    - merged_df (pd.DataFrame): DataFrame grouped by site and commodity metadata.
    - freight_class_cols (list): List of freight class columns to apply max logic to.
    - median_sources (list): List of sources considered for median.
    - wavg_sources (list): List of sources considered for weighted average.
    - q3_sources (list): List of sources considered for Q3.
    - multiplier (float): Multiplier to apply to max values (default is 1.06).

    Returns:
    - pd.DataFrame: Final DataFrame with appended recommended and ranked rows.
    """
    output_rows = []

    group_keys = ['site_description', 'site', 'unit', 'unitclass', 'commodity_group']

    for group_key, group_df in merged_df.groupby(group_keys):
        output_rows.append(group_df)

        has_any_valid_source = group_df["source"].isin(
            median_sources + wavg_sources + q3_sources
        ).any()
        if not has_any_valid_source:
            continue

        # Subsets
        median_df = group_df[group_df["source"].isin(median_sources)]
        wavg_df = group_df[group_df["source"].isin(wavg_sources)]
        q3_df = group_df[group_df["source"].isin(q3_sources)]

        # Base template
        base = group_df.iloc[0].copy()
        base[:] = None

        def make_row(label, df_subset):
            r = base.copy()
            r["source"] = label
            if not df_subset.empty:
                r["statistic"] = df_subset["statistic"].max() * multiplier
                for col in freight_class_cols:
                    r[col] = df_subset[col].max() * multiplier
            for col, val in zip(group_keys, group_key):
                r[col] = val
            return r

        # Step 1: Base recommended rows
        row_median = make_row("recommended_median", median_df)
        row_wavg = make_row("recommended_wavg", wavg_df)
        row_q3 = make_row("recommended_q3", q3_df)

        output_rows.append(pd.DataFrame([row_median, row_wavg, row_q3]))

        # Step 2: Ranked rows based on ppt_rates
        recommended_values = [
            row_median["statistic"],
            row_wavg["statistic"],
            row_q3["statistic"]
        ]

        if all(pd.notna(recommended_values)):
            sorted_values = sorted(recommended_values)
            ranked_labels = ["recommended_min", "recommended_middle", "recommended_max"]

            def make_rank_row(label, value):
                r = base.copy()
                r["source"] = label
                r["statistic"] = value
                for col, val in zip(group_keys, group_key):
                    r[col] = val
                return r

            output_rows.append(pd.DataFrame([
                make_rank_row(ranked_labels[0], sorted_values[0]),
                make_rank_row(ranked_labels[1], sorted_values[1]),
                make_rank_row(ranked_labels[2], sorted_values[2]),
            ]))

    final_df = pd.concat(output_rows, ignore_index=True)
    print("✅ Final DataFrame includes full recommendations and ranked (min/middle/max) recommendations.")
    return final_df


## Stat Excel Tab

In [61]:
# 📌 Perform data manipulation or transformation
summary_site_commodity_all = summarize_invoice_data(invoice_all_df, [
    "site", "invoice_commodity_group",
],    column_prefix="all_states_"
)
summary_site_commodity_all = summary_site_commodity_all.reset_index()
summary_site_commodity_all

,site,invoice_commodity_group,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,SPJ,1CBL,0.485000,0.698278,0.723750,1.336381,0.698256,0.497776
1,SPJ,1CPT,0.824478,1.151423,1.370540,2.680562,1.149648,0.770460
2,SPJ,1VNL,0.208449,0.573336,0.155884,0.602859,0.097703,0.106064
3,SPN,1CBL,0.355700,0.537150,0.854848,1.886769,0.799596,0.222275
4,SPN,1CPT,0.604758,1.363484,0.789551,2.213420,0.611734,0.433701
5,SPN,1VNL,0.185951,0.465604,0.163261,0.354539,0.075383,0.055487
6,SPT,1CBL,0.770815,1.568141,1.017568,1.345125,1.213528,0.664932
7,SPT,1CPT,0.870251,1.718213,1.390615,2.176484,1.317385,0.875873
8,SPT,1VNL,0.231243,0.453375,0.160061,0.427918,0.134574,0.130463
9,SPW,1CBL,0.534366,1.174034,1.056583,1.636838,0.635219,0.386252


In [62]:
# 📌 Perform data manipulation or transformation
summary_site_commodity_georgia = summarize_invoice_data(invoice_georgia_df, [
    "site", "invoice_commodity_group",
],    column_prefix="georgia_"
)
summary_site_commodity_georgia = summary_site_commodity_georgia.reset_index()
summary_site_commodity_georgia

,site,invoice_commodity_group,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg
0,SPJ,1CBL,0.700063,1.231899,1.110548,2.142453,1.031672,0.532134
1,SPJ,1CPT,0.824475,1.139937,1.365688,2.412391,1.149326,0.762656
2,SPJ,1VNL,0.161562,0.319805,0.144476,0.298258,0.087166,0.100289
3,SPN,1CBL,0.355700,0.491900,0.770000,0.878353,0.301488,0.180090
4,SPN,1CPT,0.604757,1.537428,0.780019,1.779642,0.510294,0.421856
5,SPN,1VNL,0.129432,0.410487,0.149805,0.314605,0.050975,0.047113
6,SPT,1CBL,0.808533,1.555106,1.007742,1.118137,1.044585,0.710055
7,SPT,1CPT,0.870253,1.710838,1.383988,1.823827,1.304390,0.902372
8,SPT,1VNL,0.205300,0.253647,0.153237,0.227035,0.111543,0.126457
9,SPW,1CBL,0.527304,1.212800,1.053050,1.387875,0.580463,0.381095


In [63]:
# 📌 Perform data manipulation or transformation
summary_site_commodity_combined = pd.concat([summary_site_commodity_georgia, summary_site_commodity_all], ignore_index=False).reset_index()
summary_site_commodity_combined

,index,site,invoice_commodity_group,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,0,SPJ,1CBL,0.700063,1.231899,1.110548,2.142453,1.031672,0.532134,NaN,NaN,NaN,NaN,NaN,NaN
1,1,SPJ,1CPT,0.824475,1.139937,1.365688,2.412391,1.149326,0.762656,NaN,NaN,NaN,NaN,NaN,NaN
2,2,SPJ,1VNL,0.161562,0.319805,0.144476,0.298258,0.087166,0.100289,NaN,NaN,NaN,NaN,NaN,NaN
3,3,SPN,1CBL,0.355700,0.491900,0.770000,0.878353,0.301488,0.180090,NaN,NaN,NaN,NaN,NaN,NaN
4,4,SPN,1CPT,0.604757,1.537428,0.780019,1.779642,0.510294,0.421856,NaN,NaN,NaN,NaN,NaN,NaN
5,5,SPN,1VNL,0.129432,0.410487,0.149805,0.314605,0.050975,0.047113,NaN,NaN,NaN,NaN,NaN,NaN
6,6,SPT,1CBL,0.808533,1.555106,1.007742,1.118137,1.044585,0.710055,NaN,NaN,NaN,NaN,NaN,NaN
7,7,SPT,1CPT,0.870253,1.710838,1.383988,1.823827,1.304390,0.902372,NaN,NaN,NaN,NaN,NaN,NaN
8,8,SPT,1VNL,0.205300,0.253647,0.153237,0.227035,0.111543,0.126457,NaN,NaN,NaN,NaN,NaN,NaN
9,9,SPW,1CBL,0.527304,1.212800,1.053050,1.387875,0.580463,0.381095,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
summary_site_commodity_combined.to_excel("21062025_summary_site_commodity_combined_v1300.xlsx")

In [65]:
# Step 1: Unpivot the DataFrame (wide → long)
df = summary_site_commodity_combined
id_vars = ['site', 'invoice_commodity_group', ]
df_long = df.melt(
    id_vars=id_vars,
    var_name='measure_metric',
    value_name='value'
)

# Step 2: Split 'measure_metric' column into 'measure' and 'metric'
df_long[['measure', 'metric']] = df_long['measure_metric'].str.rsplit('_', n=1, expand=True)
# Step 3: Pivot back to wide format with one row per site + commodity + unit + measure
df_result = df_long.pivot_table(
    index=['site', 'invoice_commodity_group', 'measure'],
    columns='metric',
    values='value'
).reset_index()
# Step 4: Rename for clarity (optional)
df_result = df_result.rename(columns={
    'invoice_commodity_group': 'Commodity',
   
    'median': 'median',
    'wavg': 'weighted average',
    'q3': 'Q3'
})

# Step 5: Optional sorting for cleaner Excel-style layout
df_result = df_result.sort_values(by=['site', 'Commodity', 'measure'])
df_result

metric,site,Commodity,measure,median,Q3,weighted average
0,SPJ,1CBL,all_states_historical_market,0.723750,1.336381,0.698256
1,SPJ,1CBL,all_states_historical_xgs,0.485000,0.698278,0.497776
2,SPJ,1CBL,georgia_historical_market,1.110548,2.142453,1.031672
3,SPJ,1CBL,georgia_historical_xgs,0.700063,1.231899,0.532134
4,SPJ,1CPT,all_states_historical_market,1.370540,2.680562,1.149648
5,SPJ,1CPT,all_states_historical_xgs,0.824478,1.151423,0.770460
6,SPJ,1CPT,georgia_historical_market,1.365688,2.412391,1.149326
7,SPJ,1CPT,georgia_historical_xgs,0.824475,1.139937,0.762656
8,SPJ,1VNL,all_states_historical_market,0.155884,0.602859,0.097703
9,SPJ,1VNL,all_states_historical_xgs,0.208449,0.573336,0.106064


In [66]:
df_result.to_excel("21062025_summary_site_commodity_combined_reshaped_v1300.xlsx")

In [67]:
# Start from your cleaned result table
base_df = df_result.copy()

# Group by site and commodity
recommended_rows = []

group_cols = ['site', 'Commodity']

for (site, commodity), group in base_df.groupby(group_cols):
    # Compute per-metric maximums
    max_median = group['median'].max() * 1.06
    max_q3 = group['Q3'].max() * 1.06
    max_wavg = group['weighted average'].max() * 1.06

    recommended_rows.append({
        'site': site,
        'Commodity': commodity,
        'measure': 'recommended rate',
        'median': max_median,
        'Q3': max_q3,
        'weighted average': max_wavg
    })

# Append to original data
df_final = pd.concat([base_df, pd.DataFrame(recommended_rows)], ignore_index=True)

# Sort for clean output
df_final = df_final.sort_values(by=['site', 'Commodity', 'measure'])

# Show result
df_final

,site,Commodity,measure,median,Q3,weighted average
0,SPJ,1CBL,all_states_historical_market,0.723750,1.336381,0.698256
1,SPJ,1CBL,all_states_historical_xgs,0.485000,0.698278,0.497776
2,SPJ,1CBL,georgia_historical_market,1.110548,2.142453,1.031672
3,SPJ,1CBL,georgia_historical_xgs,0.700063,1.231899,0.532134
48,SPJ,1CBL,recommended rate,1.177181,2.271001,1.093572
4,SPJ,1CPT,all_states_historical_market,1.370540,2.680562,1.149648
5,SPJ,1CPT,all_states_historical_xgs,0.824478,1.151423,0.770460
6,SPJ,1CPT,georgia_historical_market,1.365688,2.412391,1.149326
7,SPJ,1CPT,georgia_historical_xgs,0.824475,1.139937,0.762656
49,SPJ,1CPT,recommended rate,1.452772,2.841396,1.218627


In [68]:
df_final.to_excel("result_stats_summary_output_recommended_rates.xlsx")

## Site commodity freight class statistics

In [69]:
# 📌 Perform data manipulation or transformation
summary_georgia = summarize_invoice_data(invoice_georgia_df, [
    "site", "rate_unit", "invoice_commodity_group", "freight_class"
],    column_prefix="georgia_"
)
summary_georgia


georgia_historical_xgs_median  \
site rate_unit invoice_commodity_group freight_class                                  
SPJ  CWT       1VNL                    10M                                 0.084880   
                                       1M                                  0.161562   
                                       20M                                 0.084880   
                                       2M                                  0.129970   
                                       30M                                 0.064204   
...                                                                             ...   
SPW  SQYD      1CPT                    2M                                  0.805809   
                                       3M                                  0.519059   
                                       5C                                  0.833136   
                                       5M                                  0.407181   
                                       L5C                                 0.853737   

                                                      georgia_historical_xgs_q3  \
site rate_unit invoice_commodity_group freight_class                              
SPJ  CWT       1VNL                    10M                             0.084880   
                                       1M                              0.161564   
                                       20M                             0.084880   
                                       2M                              0.129971   
                                       30M                             0.064204   
...                                                                         ...   
SPW  SQYD      1CPT                    2M                              0.805809   
                                       3M                              0.627762   
                                       5C                              0.833140   
                                       5M                              0.420097   
                                       L5C                             1.924126   

                                                      georgia_historical_market_median  \
site rate_unit invoice_commodity_group freight_class                                     
SPJ  CWT       1VNL                    10M                                    0.075766   
                                       1M                                     0.126517   
                                       20M                                    0.066257   
                                       2M                                     0.102326   
                                       30M                                    0.047456   
...                                                                                ...   
SPW  SQYD      1CPT                    2M                                     0.993678   
                                       3M                                     0.539626   
                                       5C                                     1.003434   
                                       5M                                     0.618576   
                                       L5C                                    1.403939   

                                                      georgia_historical_market_q3  \
site rate_unit invoice_commodity_group freight_class                                 
SPJ  CWT       1VNL                    10M                                0.077768   
                                       1M                                 0.151995   
                                       20M                                0.073372   
                                       2M                                 0.146110   
                                       30M                                0.047456   
...                                                                  

In [70]:
# 📌 Perform data manipulation or transformation
summary_sample = summarize_invoice_data(invoice_all_df, [
    "site", "rate_unit", "invoice_commodity_group", "freight_class"
],    column_prefix="all_states_"
)
summary_sample

all_states_historical_xgs_median  \
site rate_unit invoice_commodity_group freight_class                                     
SPJ  CWT       1VNL                    10M                                    0.084880   
                                       1M                                     0.161562   
                                       20M                                    0.084880   
                                       2M                                     0.129970   
                                       30M                                    0.064204   
...                                                                                ...   
SPW  SQYD      1CPT                    2M                                     0.805809   
                                       3M                                     0.519059   
                                       5C                                     0.833136   
                                       5M                                     0.407181   
                                       L5C                                    0.853737   

                                                      all_states_historical_xgs_q3  \
site rate_unit invoice_commodity_group freight_class                                 
SPJ  CWT       1VNL                    10M                                0.084880   
                                       1M                                 0.161563   
                                       20M                                0.084880   
                                       2M                                 0.129971   
                                       30M                                0.064204   
...                                                                            ...   
SPW  SQYD      1CPT                    2M                                 0.805809   
                                       3M                                 0.627762   
                                       5C                                 0.833139   
                                       5M                                 0.420097   
                                       L5C                                2.021333   

                                                      all_states_historical_market_median  \
site rate_unit invoice_commodity_group freight_class                                        
SPJ  CWT       1VNL                    10M                                       0.076877   
                                       1M                                        0.112048   
                                       20M                                       0.071310   
                                       2M                                        0.101199   
                                       30M                                       0.047456   
...                                                                                   ...   
SPW  SQYD      1CPT                    2M                                        0.993678   
                                       3M                                        0.539626   
                                       5C                                        1.006964   
                                       5M                                        0.618576   
                                       L5C                                       1.414933   

                                                      all_states_historical_market_q3  \
site rate_unit invoice_commodity_group freight_class                                    
SPJ  CWT       1VNL                    10M                                   0.077768   
                                       1M                                    0.147036   
                                       20M                                   0.079559   
                                       2M                                    0.144827   
                    

## Create block

In [71]:
# 📌 Perform data manipulation or transformation
summary = pd.concat([summary_georgia, summary_sample], ignore_index=False).reset_index()
summary 


,site,rate_unit,invoice_commodity_group,freight_class,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,SPJ,CWT,1VNL,10M,0.084880,0.084880,0.075766,0.077768,0.057670,0.084880,NaN,NaN,NaN,NaN,NaN,NaN
1,SPJ,CWT,1VNL,1M,0.161562,0.161564,0.126517,0.151995,0.152824,0.161562,NaN,NaN,NaN,NaN,NaN,NaN
2,SPJ,CWT,1VNL,20M,0.084880,0.084880,0.066257,0.073372,0.063094,0.084751,NaN,NaN,NaN,NaN,NaN,NaN
3,SPJ,CWT,1VNL,2M,0.129970,0.129971,0.102326,0.146110,0.121498,0.129970,NaN,NaN,NaN,NaN,NaN,NaN
4,SPJ,CWT,1VNL,30M,0.064204,0.064204,0.047456,0.047456,0.047456,0.064204,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,SPW,SQYD,1CPT,2M,NaN,NaN,NaN,NaN,NaN,NaN,0.805809,0.805809,0.993678,1.051058,0.902682,0.790090
151,SPW,SQYD,1CPT,3M,NaN,NaN,NaN,NaN,NaN,NaN,0.519059,0.627762,0.539626,0.953693,0.678873,0.528425
152,SPW,SQYD,1CPT,5C,NaN,NaN,NaN,NaN,NaN,NaN,0.833136,0.833139,1.006964,1.319077,1.076278,0.833136
153,SPW,SQYD,1CPT,5M,NaN,NaN,NaN,NaN,NaN,NaN,0.407181,0.420097,0.618576,0.859560,0.656591,0.405916


In [72]:


# Generate four blocks

# Historical Georgia Mills

# Pivot each metric using safe_pivot(metric_column_name, source_name)
# Ordered: median → wavg → q3 per group

# Georgia — historical market
georgia_market_median = safe_pivot("georgia_historical_market_median", "georgia_historical_market_median",summary)
georgia_market_wavg = safe_pivot("georgia_historical_market_wavg", "georgia_historical_market_wavg",summary)
georgia_market_q3 = safe_pivot("georgia_historical_market_q3", "georgia_historical_market_q3",summary)

# Georgia — xgs
georgia_xgs_median = safe_pivot("georgia_historical_xgs_median", "georgia_historical_xgs_median",summary)
georgia_xgs_wavg = safe_pivot("georgia_historical_xgs_wavg", "georgia_historical_xgs_wavg",summary)
georgia_xgs_q3 = safe_pivot("georgia_historical_xgs_q3", "georgia_historical_xgs_q3",summary)

# All states — historical market
all_market_median = safe_pivot("all_states_historical_market_median", "all_states_historical_market_median",summary)
all_market_wavg = safe_pivot("all_states_historical_market_wavg", "all_states_historical_market_wavg",summary)
all_market_q3 = safe_pivot("all_states_historical_market_q3", "all_states_historical_market_q3",summary)



# Combine all in desired order
combined_output = pd.concat([
    georgia_market_median,
    georgia_market_wavg,
    georgia_market_q3,

    georgia_xgs_median,
    georgia_xgs_wavg,
    georgia_xgs_q3,

    all_market_median,
    all_market_wavg,
    all_market_q3,


], ignore_index=True)


# Sort for visual clarity
combined_output = combined_output.sort_values(by=["commodity_group", "site", "unit", "source"]).reset_index(drop=True)

# Preview
print("✅ Combined pivot output (clean format):")
combined_output.head()


✅ Combined pivot output (clean format):


freight_class,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
2,SPJ,SPJ,SQYD,Area,1CBL,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median
4,SPJ,SPJ,SQYD,Area,1CBL,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3


In [73]:
combined_output = prepare_freight_dataframe(combined_output, freight_classes, ordered_cols)


In [74]:
combined_output.head(2)

freight_class,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3


## Get XGS dicounted rates an combine with modelled rates

In [75]:
# Step 4: Add Source Column and Append to Vendor Data

# Load vendor data
vendor_path = "freight_rates_operating_multi_reporting_all.csv"  # Update path if needed
vendor_df = pd.read_csv(vendor_path)

# Filter vendor freight rates to selected sites
vendor_df = vendor_df[vendor_df["site"].isin(selected_sites)]


# Add source tag
vendor_df["source"] = "vendor"

vendor_df = prepare_freight_dataframe(vendor_df, freight_classes, ordered_cols, source_label="vendor")



In [76]:
# List of freight class columns
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
# Apply adjustments to each freight class column
for col in freight_classes:
    if col in vendor_df.columns:
        vendor_df[col] = adjust_rate(pd.to_numeric(vendor_df[col], errors='coerce'))

print("✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.")

✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.


In [77]:
# Step 6: Normalize Vendor Rates from $/CWT to $/LBS

# Identify rows where unit is CWT (used for 1VNL)
vendor_cwt_mask = (vendor_df["commodity_group"] == "1VNL") & (vendor_df["unit"] == "CWT")

# List of freight class columns to scale
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Convert vendor rates from $/CWT to $/LBS
vendor_df.loc[vendor_cwt_mask, freight_class_cols] = vendor_df.loc[vendor_cwt_mask, freight_class_cols] / 100

print("✅ Converted vendor CWT rates to $/LBS for comparability.")

✅ Converted vendor CWT rates to $/LBS for comparability.


In [78]:
vendor_df.to_excel('21062025_adjusted_freight_rates_table_v1300.xlsx')

In [29]:
# Append invoice summary blocks to vendor table
combined_df = pd.concat([vendor_df, combined_output], ignore_index=True)

# Preview the result
print("✅ Appended Final Table (Step 4):")
combined_df.tail()


✅ Appended Final Table (Step 4):


,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
123,SPW,SPW,CWT,Weight,1VNL,5.403926,0.214255,0.210814,0.142978,0.150177,0.140378,0.140817,0.108261,0.093772,NaN,georgia_historical_market_q3
124,SPW,SPW,CWT,Weight,1VNL,0.662675,0.177937,0.177991,0.122343,0.107951,0.096320,0.350120,0.087496,0.093772,NaN,georgia_historical_market_wavg
125,SPW,SPW,CWT,Weight,1VNL,0.836310,0.231244,0.179358,0.141366,0.141367,0.112974,0.094178,0.080727,0.058629,NaN,georgia_historical_xgs_median
126,SPW,SPW,CWT,Weight,1VNL,2.903089,0.231248,0.179359,0.141367,0.141367,0.112974,0.094178,0.089224,0.058629,NaN,georgia_historical_xgs_q3
127,SPW,SPW,CWT,Weight,1VNL,0.607491,0.231245,0.179358,0.141366,0.141367,0.112973,0.094178,0.083276,0.058629,NaN,georgia_historical_xgs_wavg


In [30]:

# convert the rates from $/LBS to $/SQFT
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

combined_df = scale_freight_rates_by_condition(
    df=combined_df,
    commodity="1VNL",
    unit="CWT",
    freight_class_cols=freight_class_cols,
    multiplier=1.2
)


✅ Scaled 1VNL with unit CWT by 1.2 for freight classes.


## Get sample size per freight class

In [ ]:
# freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# combined_df = append_invoice_freight_matrix(
#     invoice_df=invoice_all_df,
#     combined_df=combined_df,
#     freight_classes=freight_classes,
#     site_description="Itasca"  # or make it dynamic if needed
# )


## Get stast for rates table

In [39]:
summary_site_commodity_combined.head(2)

,index,site,invoice_commodity_group,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,0,SPJ,1CBL,0.700063,1.231899,1.110548,2.142453,1.031672,0.532134,NaN,NaN,NaN,NaN,NaN,NaN
1,1,SPJ,1CPT,0.824475,1.139937,1.365688,2.412391,1.149326,0.762656,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Extract relevant subset
small_table = summary_site_commodity_combined[['site', 'invoice_commodity_group',
       'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3']].copy()

# Melt into long format
small_table_long = small_table.melt(
    id_vars=["site", "invoice_commodity_group"],
    value_vars=[
        'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3'
    ],
    var_name="source",
    value_name="statistic"
)

# ✅ Rename the column
small_table_long = small_table_long.rename(columns={"invoice_commodity_group": "commodity_group"})

# Preview
small_table_long.head().sort_values(["statistic"], ascending=True)


,site,commodity_group,source,statistic
0,SPJ,1CBL,all_states_historical_market_median,NaN
1,SPJ,1CPT,all_states_historical_market_median,NaN
2,SPJ,1VNL,all_states_historical_market_median,NaN
3,SPN,1CBL,all_states_historical_market_median,NaN
4,SPN,1CPT,all_states_historical_market_median,NaN


In [41]:
small_table_long = small_table_long.dropna(subset=["statistic"])


In [42]:
small_table_long.to_excel("small_table_long2.xlsx", index=False)

In [43]:
# 📌 Merge DataFrames to combine site-level and vendor-level rate data
merged_df = pd.merge(
    combined_df,
    small_table_long,
    on=["site", "commodity_group", "source"],
    how="left"
)
merged_df

,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source,statistic
0,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
1,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
2,Jacksonville,SPJ,CWT,Weight,1VNL,0.304861,0.277934,0.215416,0.173292,0.173292,0.135835,0.113173,0.113173,0.113173,0.113173,vendor,NaN
3,Jacksonville,SPJ,SQYD,Area,1CBL,0.484997,0.472000,0.465446,0.455670,0.439341,0.439341,0.439341,0.439341,0.439341,0.439341,vendor,NaN
4,Jacksonville,SPJ,SQYD,Area,1CPT,0.824472,0.802366,0.791258,0.774706,0.746935,0.746935,0.746935,0.746935,0.746935,0.746935,vendor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,SPW,SPW,CWT,Weight,1VNL,6.484711,0.257106,0.252977,0.171573,0.180213,0.168453,0.168981,0.129913,0.112527,NaN,georgia_historical_market_q3,0.424160
124,SPW,SPW,CWT,Weight,1VNL,0.795210,0.213524,0.213589,0.146812,0.129541,0.115584,0.420144,0.104996,0.112527,NaN,georgia_historical_market_wavg,0.160926
125,SPW,SPW,CWT,Weight,1VNL,1.003572,0.277493,0.215229,0.169640,0.169640,0.135568,0.113013,0.096872,0.070355,NaN,georgia_historical_xgs_median,0.231247
126,SPW,SPW,CWT,Weight,1VNL,3.483707,0.277498,0.215230,0.169641,0.169641,0.135569,0.113014,0.107069,0.070355,NaN,georgia_historical_xgs_q3,0.615638


In [46]:
final_df = generate_recommendation_rows(
    merged_df=merged_df,
    freight_class_cols=['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M'],
    median_sources=[
        "georgia_historical_market_median",
        "all_states_historical_market_median",
        "georgia_historical_xgs_median"
    ],
    wavg_sources=[
        "georgia_historical_market_wavg",
        "all_states_historical_market_wavg",
        "georgia_historical_xgs_wavg"
    ],
    q3_sources=[
        "georgia_historical_market_q3",
        "all_states_historical_market_q3",
        "georgia_historical_xgs_q3"
    ]
)


✅ Final DataFrame includes full recommendations and ranked (min/middle/max) recommendations.


In [47]:
def classify_metric(source):
    s = source.lower()
    if 'median' in s:
        return 'median'
    elif 'wavg' in s or 'avg' in s:
        return 'weighted average'
    elif 'q3' in s:
        return 'Q3'
    elif 'vendor' in s:
        return 'baseline'
    else:
        return 'unknown'

# Apply to your DataFrame
final_df['metric'] = final_df['source'].apply(classify_metric)


In [48]:
final_df['source'].unique()

array(['vendor', 'all_states_historical_market_median',
       'all_states_historical_market_q3',
       'all_states_historical_market_wavg',
       'georgia_historical_market_median', 'georgia_historical_market_q3',
       'georgia_historical_market_wavg', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_q3', 'georgia_historical_xgs_wavg',
       'recommended_median', 'recommended_wavg', 'recommended_q3',
       'recommended_min', 'recommended_middle', 'recommended_max'],
      dtype=object)

In [49]:
# Final Sort: Enforce output row order for readability

# Define source display order
source_order = {
    "vendor": 1,

    # Georgia - historical market
    "georgia_historical_market_median": 2,
    "georgia_historical_market_wavg": 3,
    "georgia_historical_market_q3": 4,

    # All states - historical market
    "all_states_historical_market_median": 5,
    "all_states_historical_market_wavg": 6,
    "all_states_historical_market_q3": 7,

    # Georgia - xgs
    "georgia_historical_xgs_median": 8,
    "georgia_historical_xgs_wavg": 9,
    "georgia_historical_xgs_q3": 10,

    # All states - xgs
    "all_states_historical_xgs_median": 11,
    "all_states_historical_xgs_wavg": 12,
    "all_states_historical_xgs_q3": 13,

       # All states - xgs
    "recommended_median": 14,
    "recommended_wavg": 15,
    "recommended_q3": 16,
         # All states - xgs
    "recommended_min": 17,
    "recommended_middle": 18,
    "recommended_max": 19,

    "invoice_counts": 99  # Always last
}

# Add sorting key column
final_df["source_sort"] = final_df["source"].map(source_order)

# Sort rows to follow commodity hierarchy and defined source order
final_df = final_df.sort_values(
    by=["commodity_group", "site", "unit", "source_sort"]
).drop(columns="source_sort")

# Reset index for cleanliness
final_df.reset_index(drop=True, inplace=True)

print("✅ Rows sorted for visual clarity.")
display(final_df.head(20))  # Display first 20 rows for quick check


✅ Rows sorted for visual clarity.


,site_description,site,unit,unitclass,commodity_group,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source,statistic,metric
0,Jacksonville,SPJ,SQYD,Area,1CBL,0.484997,0.472000,0.465446,0.455670,0.439341,0.439341,0.439341,0.439341,0.439341,0.439341,vendor,NaN,baseline
1,SPJ,SPJ,SQYD,Area,1CBL,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median,1.110548,median
2,SPJ,SPJ,SQYD,Area,1CBL,1.243128,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_wavg,1.031672,weighted average
3,SPJ,SPJ,SQYD,Area,1CBL,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3,2.142453,Q3
4,SPJ,SPJ,SQYD,Area,1CBL,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median,0.723750,median
5,SPJ,SPJ,SQYD,Area,1CBL,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg,0.698256,weighted average
6,SPJ,SPJ,SQYD,Area,1CBL,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3,1.336381,Q3
7,SPJ,SPJ,SQYD,Area,1CBL,0.811437,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_median,0.700063,median
8,SPJ,SPJ,SQYD,Area,1CBL,0.700245,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_wavg,0.532134,weighted average
9,SPJ,SPJ,SQYD,Area,1CBL,1.385038,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_q3,1.231899,Q3


In [50]:
final_df.to_csv("output/21062025_freight_rates_nigel_v1300.csv", index=False)

## Final rates table

In [ ]:
# 🔄 Save combined_df with each site as a separate Excel sheet

import pandas as pd

# Set export path
output_path = "output/21062025_freight_rates_by_site_nigel_v1300.xlsx"  # Change path if needed

# Get unique sites
sites = final_df["site"].dropna().unique()

# Export to Excel with one sheet per site
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for site in sites:
        sheet_name = str(site)[:31]  # Excel sheet names must be ≤ 31 characters
        site_df = final_df[final_df["site"] == site]
        site_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Exported to {output_path} with one sheet per site.")
